# Lab 5 — Share knowledge with Foundry IQ

**Required · 60 minutes · Level 200**

On Day 1 you gave an agent knowledge twice, in two different ways, and both had a catch.

In Lab 2 you pasted the API reference straight into the instructions. Every request carried every fact, and changing one number meant creating a new agent version. In Lab 3 you attached a Search index, which fixed both problems — but that index belonged to that agent's tool definition. A second agent needing the same content needed its own copy of the same configuration.

**Foundry IQ** removes the copy. A **knowledge base** is a resource in its own right: you configure it once, point any number of agents at it, and change it without touching a single agent.

| | Facts in instructions (Lab 2) | Search tool on the agent (Lab 3) | Knowledge base (this lab) |
|---|---|---|---|
| Where the content lives | In the agent version | In an index, named by the agent | In a knowledge base resource |
| Sent on every request | Every fact | Only what was retrieved | Only what was retrieved |
| Updating a fact | New agent version | Reindex | Reindex |
| Two agents, same content | Duplicate the text | Duplicate the tool config | Point both at one knowledge base |
| Sources per lookup | One block of text | One index | Several, chosen per question |

That last row is the part worth slowing down for. A knowledge base can hold several **knowledge sources** — a Search index, a blob container, a SharePoint site — and decide *per question* which ones to consult. Foundry IQ calls this **agentic retrieval**: it breaks a question into subqueries, runs them in parallel across the relevant sources, reranks what comes back, and returns a grounded answer with citations.

```text
knowledge source (API reference)  \
                                   >-- knowledge base -- MCP --> agent
knowledge source (incident runbook)/
```

In this lab you will:

1. Attach a knowledge base to an agent as an MCP tool.
2. Ask a question answerable from one source, and one that needs both.
3. Ask something the knowledge base does not cover, and get an honest answer.
4. Point a second, differently-worded agent at the same knowledge base.

## New words

- **Knowledge base** — the resource an agent queries. It names its sources and the rules for searching them. Several agents can share one.
- **Knowledge source** — one place content actually lives: a Search index, a blob container, a SharePoint site, a website.
- **Agentic retrieval** — Foundry IQ's retrieval strategy: split the question into subqueries, run them across the relevant sources at once, rerank, then answer with citations.
- **MCP** — Model Context Protocol, an open standard that lets an agent call tools hosted somewhere else over HTTP. A knowledge base is reached this way.
- **MCP server** — the endpoint on the other side of that protocol. For Foundry IQ it is your Search service, at `/knowledgebases/<name>/mcp`.
- **Project connection** — a stored entry in your Foundry project holding the address and sign-in details for an external service, so an agent never carries a credential.

### Why does a knowledge base speak MCP?

It seems like a detour: Foundry IQ is a Microsoft feature, so why reach it through an open protocol instead of a dedicated knowledge-base tool class?

Because MCP is how Foundry talks to *anything* hosted elsewhere. Learning this one wiring pattern — connection, then `MCPTool`, then attach — is what lets you connect a ticketing system, an internal API, or a partner service later. The knowledge base is simply the friendliest first example of it.

## Before you start

- Labs 2 and 3 finished, so `create_version`, `agent_reference` and Search indexes are familiar.
- Python 3.11 or later, `az login` completed, and a notebook kernel selected.
- A Foundry project endpoint and a model deployment.
- An Azure AI Search service holding a knowledge base, and a Foundry project connection pointing at it. The next section shows exactly how both are built.

**How the To-Do sections work.** Replace each `...` blank, then run the cell with **Shift+Enter**. A blank you leave open stops the cell with a message naming it, before anything reaches Azure. Try the task, then the hint, then the solution.

<details><summary><b>How the knowledge base is built</b> — expand this if you are setting up from scratch</summary>

Two things must exist before the notebook runs, and neither is created with `azure-ai-projects`. That is the first surprising thing about Foundry IQ, and worth knowing before you go looking for an API that is not there.

**1. Knowledge sources and a knowledge base, on the Search service.** These are Azure AI Search objects, so they come from `azure-search-documents`:

```python
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import KnowledgeBase, KnowledgeSourceReference

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

knowledge_base = KnowledgeBase(
    name="orders-kb",
    description="Answers questions about the orders API and about handling its incidents.",
    knowledge_sources=[
        KnowledgeSourceReference(name="orders-docs-ks"),
        KnowledgeSourceReference(name="orders-runbook-ks"),
    ],
    encryption_key=None,
)
index_client.create_or_update_knowledge_base(knowledge_base)
```

A knowledge source and its knowledge base must live on the same search service. Each source wraps content you have already indexed — in this case two small sets of documents, one describing the API and one describing what to do when it misbehaves.

**2. A project connection of category `RemoteTool`.** This is what lets the agent reach the search service without holding a credential. It is created through Azure Resource Manager rather than a data-plane SDK, so use the portal or an ARM call. Its `target` is the knowledge base's MCP endpoint and its `authType` is `ProjectManagedIdentity`, meaning the project's own managed identity authenticates. That identity needs read access on the Search service.

**What the two sources contain:**

| Source | Example content |
|---|---|
| `orders-docs-ks` | The rate limit is 100 requests per minute. Bearer tokens expire after 60 minutes. `page_size` may not exceed 200. |
| `orders-runbook-ks` | On sustained HTTP 429, back off using the `Retry-After` header and raise a ticket if it persists past 15 minutes. |

Nothing in either source mentions pricing, which is what makes the honesty test in section 3 meaningful.

**If Foundry IQ is unavailable** in your region or on your subscription, read this notebook rather than running it, and say plainly that the live activity was not performed. Do not substitute a plain Search index and call it Foundry IQ — the difference is exactly what this lab teaches.

</details>

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0"

## 0. Connect to your project

The next cell reads four settings. Two are the ones you used all through Day 1; two point at the knowledge base:

| Setting | What it is |
|---|---|
| `PROJECT_ENDPOINT` | Your Foundry project |
| `MODEL_DEPLOYMENT` | The model your agents run on |
| `KNOWLEDGE_BASE_MCP_URL` | The knowledge base's MCP endpoint |
| `KB_CONNECTION_NAME` | The project connection that authenticates to it |

The MCP URL is worth a second look before you move on:

```text
https://<search-service>.search.windows.net/knowledgebases/orders-kb/mcp?api-version=...
                                            ^^^^^^^^^^^^^^ ^^^^^^^^^ ^^^
                                            the resource   its name  the protocol endpoint
```

It is a plain HTTPS address ending in `/mcp`. That is the whole trick — the knowledge base is a service your agent calls over the network.

`check_todos` is a helper, not part of the exercise. It stops a cell while a `...` blank is still open.

**Run the cell. You should see** `Ready.` followed by the two agent names it reserved. If a setting is missing, the cell names it before anything reaches Azure.

In [ ]:
import os
import sys
from uuid import uuid4

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import AzureCliCredential

# Your nonsecret settings. Paste them here or set them as environment variables.
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")
KNOWLEDGE_BASE_MCP_URL = os.getenv("AZURE_AI_KNOWLEDGE_BASE_MCP_URL", "")
KB_CONNECTION_NAME = os.getenv("AZURE_AI_KNOWLEDGE_BASE_CONNECTION", "")

if sys.version_info < (3, 11):
    raise RuntimeError(
        "These notebooks need Python 3.11 or later. This kernel is "
        f"{sys.version_info.major}.{sys.version_info.minor}. Select a newer kernel."
    )

missing = [
    name
    for name, value in {
        "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT,
        "AZURE_AI_KNOWLEDGE_BASE_MCP_URL": KNOWLEDGE_BASE_MCP_URL,
        "AZURE_AI_KNOWLEDGE_BASE_CONNECTION": KB_CONNECTION_NAME,
    }.items()
    if not value
]
if missing:
    raise ValueError(f"Set these before continuing: {', '.join(missing)}")


def check_todos(**answers: object) -> None:
    """Helper. Stops the cell while a `...` blank is still open."""
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


SUPPORT_NAME = f"day2-support-{uuid4().hex[:8]}"
INCIDENT_NAME = f"day2-incident-{uuid4().hex[:8]}"

credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client(timeout=120, max_retries=0)
print(f"Ready. Agents to create: {SUPPORT_NAME} and {INCIDENT_NAME}")

## 1. Describe the knowledge base as a tool

`MCPTool` hands an agent an address and a set of rules for using it. Four fields matter here, and each answers a different question:

| Field | Question it answers | What goes wrong if you get it wrong |
|---|---|---|
| `server_label` | What is this tool called in the transcript? | Nothing breaks, but tool calls become hard to read |
| `server_url` | Where is the knowledge base? | The agent cannot reach it and the call fails |
| `project_connection_id` | How does it sign in? | `401 Unauthorized`, because no credential is attached |
| `allowed_tools` | Which operations may it call? | The agent may attempt operations you did not intend |

`allowed_tools` deserves the extra sentence. An MCP server can advertise many operations, and by default an agent may call any of them. Listing names restricts it to exactly those. A knowledge base publishes one operation for agents, **`knowledge_base_retrieve`**, and naming it explicitly is both the safe habit and a piece of documentation for the next reader.

There is a fifth field worth recognising: `require_approval`. Set to `"always"`, the service pauses and waits for your code to approve each tool call before it runs — the right choice for a tool that sends email or moves money. Retrieval only reads, so `"never"` is appropriate and keeps this lab a single round trip.

### To-Do 1 — Point the tool at the knowledge base

**Goal:** an `MCPTool` that can reach the knowledge base and is allowed to do exactly one thing.

**Steps**

1. Set `SERVER_URL` to the MCP endpoint from section 0.
2. Set `CONNECTION_ID` to the connection name, so the agent authenticates as the project rather than as you.
3. Set `ALLOWED_TOOLS` to a list containing the single retrieval operation named above.
4. Run the cell.

**Predict:** you leave `allowed_tools` out entirely. Does the lab still work, and what have you given up?

**Run the cell. You should see** `Tool configured` with the label, the host it points at, and the one allowed operation. Nothing has been created in Azure yet — this is a description, exactly like `PromptAgentDefinition` was in Lab 2.

<details><summary>Hint</summary>

Two of the three values are already in variables from section 0. The third is a string you read in the paragraph above. `ALLOWED_TOOLS` takes a **list**, even with only one entry.

</details>

<details><summary>Show solution code</summary>

```python
SERVER_URL = KNOWLEDGE_BASE_MCP_URL
CONNECTION_ID = KB_CONNECTION_NAME
ALLOWED_TOOLS = ["knowledge_base_retrieve"]
```

</details>

In [ ]:
from urllib.parse import urlparse

SERVER_URL = ...  # TODO 1: the knowledge base MCP endpoint.
CONNECTION_ID = ...  # TODO 1: the project connection that authenticates to it.
ALLOWED_TOOLS = ...  # TODO 1: a list holding the one retrieval operation.
check_todos(SERVER_URL=SERVER_URL, CONNECTION_ID=CONNECTION_ID, ALLOWED_TOOLS=ALLOWED_TOOLS)

kb_tool = MCPTool(
    server_label="orders_knowledge",
    server_url=SERVER_URL,
    project_connection_id=CONNECTION_ID,
    allowed_tools=ALLOWED_TOOLS,
    require_approval="never",
)
print(f"Tool configured: {kb_tool.server_label}")
print(f"  points at: {urlparse(SERVER_URL).hostname}")
print(f"  may call:  {', '.join(ALLOWED_TOOLS)}")

## 2. Give the tool to an agent

This is the same `create_version` call you made in Lab 2, with one addition: `tools=[kb_tool]`. The agent now has somewhere to look things up.

Having a source and *using* it are different things, though. A model that can retrieve will still answer from memory when the question feels easy — and questions about rate limits and status codes feel very easy, because the model has seen thousands of APIs. It just has not seen yours.

Three rules close that gap:

```text
1. Always retrieve      never answer from memory
2. Always cite          name the source for each fact
3. Admit the gap        say so plainly when retrieval comes back empty
```

Rule 3 is the one people leave out, and it decides whether the agent is trustworthy. Without it, a model that finds nothing will fill the silence with a plausible number. You will test that directly in the next section.

### To-Do 2 — Write the retrieval rules and create the agent

**Goal:** an agent that retrieves before answering, cites what it used, and admits what it does not know.

**Steps**

1. Write `RETRIEVAL_RULES` covering all three rules above.
2. Pass the tool to the definition with `tools=[kb_tool]`.
3. Run the cell **once**. Each run creates another version.

**Run the cell. You should see** `Created <name> version 1, with 1 tool attached`.

<details><summary>Hint</summary>

Write it as instructions to the agent, in the second person. Be specific about the empty case: "say that the knowledge base does not cover it" leaves far less room than "be helpful".

</details>

<details><summary>Show solution code</summary>

```python
RETRIEVAL_RULES = (
    "Always search the knowledge base before answering. Never answer from your own memory, "
    "even when the question resembles a common API. "
    "Cite the title of every source you used. "
    "If the knowledge base returns nothing relevant, say that it does not cover the question. "
    "Never invent a limit, an endpoint or a status code."
)
```

</details>

In [ ]:
SUPPORT_ROLE = (
    "You support developers integrating with the internal orders API. "
    "Answer their questions about how it behaves."
)

RETRIEVAL_RULES = ...  # TODO 2: retrieve first, cite sources, admit gaps.
check_todos(RETRIEVAL_RULES=RETRIEVAL_RULES)

support = project.agents.create_version(
    agent_name=SUPPORT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=f"{SUPPORT_ROLE}\n{RETRIEVAL_RULES}",
        tools=[kb_tool],  # TODO 2: hand the knowledge base to the agent.
    ),
)
print(f"Created {support.name} version {support.version}, with 1 tool attached.")

## 3. Watch retrieval decide

The helper below asks a question and then prints something you have not looked at before: the **items** in `response.output`.

A response is not a single blob of text. It is a list of things that happened, in order. When an agent uses a tool, the tool call appears in that list alongside the message:

```text
response.output  ->  [ mcp_call(server_label="orders_knowledge"), message("The limit is ...") ]
                       ^ the lookup                                 ^ the answer
```

Reading that list is how you tell retrieval from recall. If you only ever print `output_text`, an answer from memory and an answer from the knowledge base look identical — and one of them is a liability.

Three questions run next, each testing something different:

| # | Question | What it tests |
|---|---|---|
| 1 | The rate limit | One source answers it |
| 2 | Sustained 429s in production | Both sources, combined |
| 3 | Price per request | Nothing covers it |

Question 2 is the interesting one. The limit lives in the API reference and the response procedure lives in the runbook, so a single-source lookup cannot answer it completely. This is where agentic retrieval earns its name.

**Predict:** which of the three will show *no* tool call in the output list?

**Run the cell. You should see** each question, whether the knowledge base was called, and the answer. Questions 1 and 2 should be answered from retrieved content with citations. Question 3 should say the knowledge base does not cover pricing.

If question 3 produces a confident number instead, your rule 3 was too soft. Note the wording you used — a boundary rule that fails is a more useful thing to have seen than one that works first time.

In [ ]:
def ask(agent, question):
    """Ask one agent version a question and show whether it used the knowledge base."""
    response = client.responses.create(
        input=question,
        extra_body={
            "agent_reference": {
                "type": "agent_reference",
                "name": agent.name,
                "version": str(agent.version),
            }
        },
    )
    tool_calls = [item for item in response.output if getattr(item, "type", "") != "message"]
    print(f"Q: {question}")
    print(f"   knowledge base called: {'yes' if tool_calls else 'no'} ({len(tool_calls)} tool item(s))")
    print(f"   {response.output_text}\n")
    return response


single_source = ask(support, "What is the rate limit on the orders API?")
combined = ask(support, "Our client is getting sustained 429s in production. What is the limit, and what should we do about it?")
uncovered = ask(support, "How much does the orders API cost per request?")

## 4. Share the knowledge base with a second agent

Here is the claim this lab has been building towards: **the knowledge base does not belong to the support agent.**

To test it, create a second agent with a genuinely different job — an incident responder that produces an ordered checklist rather than an explanation — and hand it *the same* `kb_tool` object. No copying, no reindexing, no second configuration.

```text
                 support agent   --\
                                    >-- orders-kb
                 incident agent  --/
```

Think of it as a library card. The two agents hold different jobs and ask different questions, but they read from one library. Adding a document helps both, and neither owns the shelf.

This is why the row "two agents, same content" in the opening table matters. In Lab 3 a second agent needed its own tool configuration pointing at its own index. Here it needs neither.

### To-Do 3 — Point a second agent at the same knowledge base

**Goal:** two agents with different behaviour, reading from one knowledge base.

**Steps**

1. Write `INCIDENT_ROLE` so the incident responder has a clearly different job from the support agent. Ask for an ordered checklist, not an explanation.
2. Attach the **same** `kb_tool` — do not build a new one.
3. Run the cell and compare the two outputs. The facts should agree; the shape should not.

**Predict:** the runbook changes tomorrow morning. How many agents need changing?

**Run the cell. You should see** the incident agent created, then a checklist built from the same content the support agent quoted.

<details><summary>Hint</summary>

Keep the retrieval rules identical, so the only thing that differs is the job. That is what makes the comparison fair — the same one-variable discipline you used when comparing agent versions in Lab 2.

</details>

<details><summary>Show solution code</summary>

```python
INCIDENT_ROLE = (
    "You guide an on-call engineer through an incident with the orders API. "
    "Reply with a short numbered checklist of actions, most urgent first."
)
```

</details>

In [ ]:
INCIDENT_ROLE = ...  # TODO 3: a different job from the support agent, producing a checklist.
check_todos(INCIDENT_ROLE=INCIDENT_ROLE)

incident = project.agents.create_version(
    agent_name=INCIDENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT,
        instructions=f"{INCIDENT_ROLE}\n{RETRIEVAL_RULES}",
        tools=[kb_tool],  # TODO 3: the same tool object, not a new one.
    ),
)
print(f"Created {incident.name} version {incident.version}, sharing one knowledge base.\n")

checklist = ask(incident, "We are being throttled by the orders API right now. What should I do?")

## Deterministic success check

Model wording varies, so this checks structure. It confirms that both agents exist, that both were given exactly one tool, that the questions with coverage triggered a lookup, and that both agents point at the same knowledge base rather than at copies.

In [ ]:
assert support.name != incident.name, "The two agents should be separate resources."
for label, response in [
    ("single source", single_source),
    ("combined", combined),
    ("uncovered", uncovered),
    ("checklist", checklist),
]:
    assert response.status == "completed", f"The {label} request did not complete."
    assert response.output_text.strip(), f"The {label} response came back empty."

for label, response in [("single source", single_source), ("combined", combined), ("checklist", checklist)]:
    called = [item for item in response.output if getattr(item, "type", "") != "message"]
    assert called, (
        f"The {label} question was answered without calling the knowledge base. "
        "Strengthen the 'always search first' rule in To-Do 2."
    )

assert kb_tool.server_url == SERVER_URL, "Both agents should share the tool you built in To-Do 1."
print(
    f"PASS - {support.name} and {incident.name} both answered from one knowledge base, "
    "and each was given exactly one tool."
)

## What you learned

- A **knowledge base** is a resource, not a setting on an agent. Configure it once and share it.
- It reaches several **knowledge sources** and picks the relevant ones per question, which is what let the combined question work.
- Agents connect to it through **MCP**, the same wiring you would use for any externally hosted tool.
- `project_connection_id` is how an agent authenticates without holding a credential.
- `allowed_tools` narrows an MCP server to the operations you intended.
- Reading `response.output` is how you prove retrieval happened. `output_text` alone cannot tell you.
- Retrieval makes an honest answer *possible*; the instruction to admit gaps is what makes it *likely*.

**Reflection.** One sentence each.

1. The rate limit doubles next week. What has to change, and how many agents does it touch?
2. Your agent answers correctly but shows no tool call. Why is that a problem rather than a pleasant surprise?
3. When would you still paste facts directly into instructions, as you did in Lab 2?

<details><summary>Compare your answers</summary>

1. The API reference source is updated and reindexed. No agent changes and no version bump, because neither agent stores the number — that is the whole return on making knowledge a separate resource.
2. Because it was answered from the model's memory. It happened to be right this time; nothing guarantees the next one, and there is no citation to check. With rate limits and status codes this is a live risk, since the model has strong priors about what APIs *usually* do.
3. When the content is tiny, stable, and specific to that one agent — a tone rule, a fixed disclaimer, a format. The moment two agents need the same fact, or the fact changes on its own schedule, it belongs in a knowledge base.

</details>

**If something fails:** a `401` usually means `project_connection_id` is wrong, or the project's managed identity lacks read access on the Search service. A `404` usually means the MCP URL is wrong — check that it ends in `/mcp` and includes the API version. If a tool call is reported but no content comes back, a knowledge source may be empty or still indexing. Restart the kernel after changing packages.

**Reset:** the cleanup cell closes local clients only. To remove the two agents afterwards, delete them from the portal or with `project.agents.delete_version(...)`. The knowledge base is shared by both agents, so deleting it affects everything pointing at it.

**Expected artifact:** two agents sharing one knowledge base, and a passing success check.

**Next:** Lab 6 asks what happens when an instruction is not enough, and what a platform control does that instructions cannot.

In [ ]:
client.close()
project.close()
credential.close()
print("Closed the local clients. The knowledge base and your agents remain in Azure.")